# DocTamper AI: Cloud Streamlit Deployment on Kaggle
**Zero Local Storage Required:** Runs the full interactive web application, neural inference (PyTorch GPU/CPU), and visual UI on Kaggle's cloud servers, exposing a live public URL for presentation and viva demonstration.


## 1. Clone Project Repository
Clones the DocTamper AI codebase from GitHub or sets up the working directory.


In [ ]:
import os
import sys
import time
import subprocess
from pathlib import Path

# Clone repo if not already present
if not os.path.exists("document-tampering-detection"):
    print("Cloning repository from GitHub...")
    # Replace with your repository URL if private: https://<token>@github.com/username/repo.git
    !git clone https://github.com/titas-mahato/document-tampering-detection.git
    os.chdir("document-tampering-detection")
else:
    os.chdir("document-tampering-detection")
    !git pull

print("Current Directory:", os.getcwd())


## 2. Install Dependencies & Cloud Tunnel
Installs `streamlit`, `opencv-python-headless`, and downloads the lightweight `cloudflared` tunnel binary (no accounts or tokens needed).


In [ ]:
!pip install -q streamlit opencv-python-headless

# Download Cloudflare tunnel binary (Linux x64)
if not os.path.exists("cloudflared"):
    print("Downloading Cloudflare Tunnel binary...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared
    print("Cloudflared binary ready.")


## 3. Verify Model Weights
Checks for the presence of both trained PyTorch models:
1. `casia_dual_stream_best.pth` (Stage 1 Classifier)
2. `doctamper_resnet34_unet_best.pth` (Stage 2 U-Net Localizer)


In [ ]:
weights_dir = Path("weights")
weights_dir.mkdir(exist_ok=True)

# Search for weights in /kaggle/input if attached as a dataset
input_dir = Path("/kaggle/input")
for pth in input_dir.rglob("*.pth"):
    target = weights_dir / pth.name
    if not target.exists():
        print(f"Found weight in input: {pth.name} -> Copying to weights/")
        import shutil
        shutil.copy(pth, target)

# Also check /kaggle/working
for pth in Path("/kaggle/working").rglob("*.pth"):
    if "weights" not in str(pth):
        target = weights_dir / pth.name
        if not target.exists():
            import shutil
            shutil.copy(pth, target)

print("\nVerified weights in weights/ folder:")
for w in weights_dir.glob("*.pth"):
    print(f" - {w.name} ({w.stat().st_size / (1024*1024):.1f} MB)")


## 4. Launch Streamlit & Start Live Cloud Tunnel
Starts Streamlit in the background and opens an instant secure HTTPS tunnel via Cloudflare.


In [ ]:
# Kill any previous instances
!pkill -f streamlit
!pkill -f cloudflared

# 1. Start Streamlit server in the background
print("Starting Streamlit application on port 8501...")
!nohup streamlit run app/app.py --server.port 8501 --server.address 127.0.0.1 > streamlit.log 2>&1 &

time.sleep(3)

# 2. Start Cloudflare Tunnel
print("Starting Cloudflare Tunnel...")
!nohup ./cloudflared tunnel --url http://127.0.0.1:8501 > tunnel.log 2>&1 &

# Wait for tunnel URL to generate
time.sleep(5)

# Extract and print public URL
tunnel_url = None
with open("tunnel.log", "r") as f:
    for line in f:
        if "trycloudflare.com" in line:
            parts = line.split("https://")
            if len(parts) > 1:
                tunnel_url = "https://" + parts[1].split()[0]
                break

print("=" * 65)
if tunnel_url:
    print("LIVE DEMO IS READY! Click the public URL below:")
    print(tunnel_url)
else:
    print("Tunnel starting... Check tunnel.log below for your URL:")
    !cat tunnel.log | grep trycloudflare
print("=" * 65)


## 5. Live App Logs & Status
Inspect runtime outputs, prediction requests, and server status.


In [ ]:
!tail -n 25 streamlit.log
